API key

In [2]:
from google.colab import userdata
API_KEY = userdata.get('OWM_API_KEY')

Geocoding + Forecast Fetch

In [3]:
import requests

def geocode_location(name, api_key, country_code="GH"):
    """Converts a place name into lat/lon using OpenWeatherMap's Geocoding API."""
    resp = requests.get(
        "https://api.openweathermap.org/geo/1.0/direct",
        params={"q": f"{name},{country_code}", "limit": 1, "appid": api_key}
    )
    results = resp.json()
    if not results:
        return None, None
    return results[0]['lat'], results[0]['lon']


def get_forecast(lat, lon, api_key):
    resp = requests.get(
        "https://api.openweathermap.org/data/2.5/forecast",
        params={"lat": lat, "lon": lon, "appid": api_key, "units": "metric"}
    )
    return resp.json()

Region Lookup Table

In [4]:
REGION_TO_TOWN = {
    "AHAFO": "Goaso",
    "ASHANTI": "Kumasi",
    "BONO": "Sunyani",
    "BONO EAST": "Techiman",
    "CENTRAL": "Cape Coast",
    "EASTERN": "Koforidua",
    "GREATER ACCRA": "Accra",
    "NORTHEAST": "Nalerigu",
    "NORTHERN": "Tamale",
    "OTI": "Dambai",
    "SAVANNAH": "Damongo",
    "UPPER EAST": "Bolgatanga",
    "UPPER WEST": "Wa",
    "VOLTA": "Ho",
    "WESTERN": "Sekondi",
    "WESTERN NORTH": "Sefwi Wiawso",
}

Resolve_Location

In [5]:
def resolve_location(location_name, api_key):
    """Accepts either a Ghana region name (mapped to its capital via
    REGION_TO_TOWN) or a specific town/city name directly."""
    town = REGION_TO_TOWN.get(location_name.upper())
    query = town if town else location_name
    return geocode_location(query, api_key)

Distance Check

In [6]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    """Straight-line distance between two coordinates, in km."""
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def validate_town_matches_region(town_name, region_name, api_key, threshold_km=100):
    town_lat, town_lon = geocode_location(town_name, api_key)
    if town_lat is None:
        return False, None, f"Could not find '{town_name}' — check the spelling."

    region_capital = REGION_TO_TOWN.get(region_name.upper())
    if region_capital is None:
        return False, None, f"'{region_name}' is not a recognized region."

    region_lat, region_lon = geocode_location(region_capital, api_key)
    distance = haversine_km(town_lat, town_lon, region_lat, region_lon)

    if distance > threshold_km:
        return False, distance, (
            f"'{town_name}' looks far from {region_name} region "
            f"(~{distance:.0f}km from {region_capital}). Please double-check."
        )
    return True, distance, None

Validation Test

In [7]:
ok, dist, msg = validate_town_matches_region("Kpando", "VOLTA", API_KEY)
print(ok, dist, msg)

ok, dist, msg = validate_town_matches_region("Ho", "VOLTA", API_KEY)
print(ok, dist, msg)

True 46.92708087726862 None
True 0.0 None


Resolve_Location Test + Fetch Real Forecast

In [17]:
lat, lon = resolve_location("GREATER ACCRA", API_KEY)
print("Region lookup:", lat, lon)

lat, lon = resolve_location(location_name="Berekuso", api_key=API_KEY)
print("Direct town:", lat, lon)

lat, lon = resolve_location("Nonsense Place", API_KEY)
print("Invalid input:", lat, lon)

# Use one real result to fetch an actual forecast for the next cells
lat, lon = resolve_location("EASTERN", API_KEY)
forecast_data = get_forecast(lat, lon, API_KEY)
print(forecast_data['city']['name'])

Region lookup: 5.5571096 -0.2012376
Direct town: 5.7593581 -0.2250359
Invalid input: None None
Koforidua


Core Weather Signals

In [9]:
from collections import defaultdict

def _daily_rain_totals(forecast_json):
    """Groups the 5-day/3-hour forecast into full-day totals. Only
    returns days with all 8 blocks present -- partial days at the start
    and end of the forecast window don't have enough data to judge fairly."""
    daily_rain = defaultdict(float)
    daily_block_count = defaultdict(int)
    for block in forecast_json['list']:
        date = block['dt_txt'].split(' ')[0]
        rain_mm = block.get('rain', {}).get('3h', 0)
        daily_rain[date] += rain_mm
        daily_block_count[date] += 1
    return {d: total for d, total in daily_rain.items() if daily_block_count[d] == 8}


def check_rain_expected(forecast_json, hours=48, pop_threshold=0.5):
    """Looks at the next `hours` worth of forecast blocks.
    Returns True if any block has a high chance of rain."""
    blocks_needed = hours // 3
    upcoming = forecast_json['list'][:blocks_needed]
    for block in upcoming:
        if block['pop'] >= pop_threshold:
            rain_mm = block.get('rain', {}).get('3h', 0)
            return True, block['dt_txt'], rain_mm
    return False, None, 0


def check_dry_spell(forecast_json, daily_threshold_mm=1.0):
    """Counts how many full days are forecasted as dry."""
    full_days = _daily_rain_totals(forecast_json)
    dry_days = [d for d, total in full_days.items() if total < daily_threshold_mm]
    return len(dry_days), full_days

Test Core Signals on Real Data

In [10]:
rain_soon, when, amount = check_rain_expected(forecast_data)
print("Rain expected in next 48h:", rain_soon, "-", when, f"({amount}mm)")

dry_count, daily_totals = check_dry_spell(forecast_data)
print("Dry days out of 5:", dry_count)
print(daily_totals)

Rain expected in next 48h: True - 2026-08-11 15:00:00 (3.16mm)
Dry days out of 5: 3
{'2026-08-10': 0.38, '2026-08-11': 4.44, '2026-08-12': 2.07, '2026-08-13': 0.71, '2026-08-14': 0.27}


Farmer-Facing Alerts

In [11]:
def get_irrigation_alert(forecast_json, dry_day_threshold=3):
    """Turns the raw dry-spell count into a farmer-facing alert."""
    dry_count, _ = check_dry_spell(forecast_json)
    if dry_count >= dry_day_threshold:
        return True, f"Irrigate now — {dry_count} dry day(s) forecasted with no significant rain."
    return False, None


def check_planting_window(forecast_json, min_moderate_days=2, heavy_mm=15.0, light_mm=1.0):
    """Flags whether the upcoming forecast looks favorable for planting:
    steady, moderate rain -- not dry, and not a downpour that risks
    washing out seeds."""
    full_days = _daily_rain_totals(forecast_json)
    moderate_days = [d for d, t in full_days.items() if light_mm <= t <= heavy_mm]
    heavy_days = [d for d, t in full_days.items() if t > heavy_mm]

    if heavy_days:
        return False, "Heavy rain expected — risk of seed washout, hold off planting."
    elif len(moderate_days) >= min_moderate_days:
        return True, f"Good window to plant — steady rain expected over the next {len(moderate_days)} day(s)."
    else:
        return False, "Too dry for planting right now — irrigate first."

Test the Alerts on Real Data

In [12]:
should_irrigate, irrigation_msg = get_irrigation_alert(forecast_data)
print("Irrigation alert:", should_irrigate, "-", irrigation_msg)

can_plant, planting_msg = check_planting_window(forecast_data)
print("Planting window:", can_plant, "-", planting_msg)

Irrigation alert: True - Irrigate now — 3 dry day(s) forecasted with no significant rain.
Planting window: True - Good window to plant — steady rain expected over the next 2 day(s).


Scenario Tests 1–3

In [13]:
def make_fake_forecast(rain_pattern):
    """rain_pattern: list of (pop, rain_mm) tuples, one per 3h block, 8 blocks/day"""
    from datetime import datetime, timedelta
    base = datetime(2026, 8, 8, 0, 0, 0)
    blocks = []
    for i, (pop, mm) in enumerate(rain_pattern):
        dt_txt = (base + timedelta(hours=3*i)).strftime('%Y-%m-%d %H:%M:%S')
        block = {'dt_txt': dt_txt, 'pop': pop}
        if mm > 0:
            block['rain'] = {'3h': mm}
        blocks.append(block)
    return {'list': blocks}

# Scenario 1: heavy rain incoming tomorrow
heavy_rain = make_fake_forecast([(0.1, 0)]*4 + [(0.9, 5.0)] + [(0.1,0)]*35)
rain_soon, when, amt = check_rain_expected(heavy_rain)
assert rain_soon == True, "Should detect incoming heavy rain"
print("Test 1 passed: heavy rain correctly detected")

# Scenario 2: bone-dry week, no rain anywhere
bone_dry = make_fake_forecast([(0.05, 0)]*40)
rain_soon, when, amt = check_rain_expected(bone_dry)
dry_count, breakdown = check_dry_spell(bone_dry)
assert rain_soon == False, "Should NOT detect rain when there's none"
assert dry_count == 5, f"All 5 full days should be dry, got {dry_count}"
print("Test 2 passed: dry week correctly identified")

# Scenario 3: light drizzle, below threshold — shouldn't trigger "rain expected"
light_drizzle = make_fake_forecast([(0.3, 0.05)]*40)
rain_soon, when, amt = check_rain_expected(light_drizzle)
assert rain_soon == False, "Low pop/light drizzle shouldn't trigger the alert"
print("Test 3 passed: light drizzle correctly ignored")

print("\nAll scenario tests passed.")

Test 1 passed: heavy rain correctly detected
Test 2 passed: dry week correctly identified
Test 3 passed: light drizzle correctly ignored

All scenario tests passed.


Scenario Tests for Irrigation + Planting

In [14]:
# Scenario 4: prolonged dry spell -- irrigation alert should fire
should_irrigate, msg = get_irrigation_alert(bone_dry)
assert should_irrigate == True, "Should recommend irrigation during a dry spell"
print("Test 4 passed:", msg)

# Scenario 5: steady moderate rain -- no irrigation needed
moderate_rain = make_fake_forecast([(0.6, 0.625)]*40)  # ~5mm/day
should_irrigate, msg = get_irrigation_alert(moderate_rain)
assert should_irrigate == False, "Should NOT recommend irrigation when rain is steady"
print("Test 5 passed: no irrigation alert during steady rain")

# Scenario 6: steady moderate rain -- good planting window
can_plant, msg = check_planting_window(moderate_rain)
assert can_plant == True, "Should recommend planting during steady moderate rain"
print("Test 6 passed:", msg)

# Scenario 7: heavy downpour -- hold off planting
heavy_downpour = make_fake_forecast([(0.9, 3.0)]*40)  # 24mm/day
can_plant, msg = check_planting_window(heavy_downpour)
assert can_plant == False, "Should NOT recommend planting during a heavy downpour"
print("Test 7 passed:", msg)

# Scenario 8: bone dry -- too dry to plant
can_plant, msg = check_planting_window(bone_dry)
assert can_plant == False, "Should NOT recommend planting when it's bone dry"
print("Test 8 passed:", msg)

print("\nAll irrigation + planting scenario tests passed.")

Test 4 passed: Irrigate now — 5 dry day(s) forecasted with no significant rain.
Test 5 passed: no irrigation alert during steady rain
Test 6 passed: Good window to plant — steady rain expected over the next 5 day(s).
Test 7 passed: Heavy rain expected — risk of seed washout, hold off planting.
Test 8 passed: Too dry for planting right now — irrigate first.

All irrigation + planting scenario tests passed.
